#### Milestone 2 part 1

In [ ]:
#=========================================
#= Importing all the neccassry library =
#==========================================
import numpy as np
import pandas as pd
from pathlib import Path
import librosa
import librosa.display
import soundfile as sf

In [ ]:
### path configuration

BASE_DIR     = Path("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup")
STEMS_DIR    = BASE_DIR / "genres_stems"
ESC50_DIR    = BASE_DIR / "ESC-50-master"
MASHUPS_DIR  = BASE_DIR / "mashups"
TEST_CSV     = BASE_DIR / "test.csv"
SAMPLE_SUB   = BASE_DIR / "sample_submission.csv"
ESC50_META   = ESC50_DIR / "meta" / "esc50.csv"

GENRES = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]
STEMS  = ["drums.wav","vocals.wav","bass.wav","other.wav"]

print("BASE_DIR exists:", BASE_DIR.exists())
print("Genres found   :", [g for g in GENRES if (STEMS_DIR/g).exists()])

In [ ]:
# What is the mean duration (in seconds) of the Jazz genre stems in train dataset?
# path for *jazz" genres
jazz_path=STEMS_DIR/"jazz"
print(" jazz-path exist",jazz_path.exists())
durations = []
songs = sorted([d for d in jazz_path.iterdir() if d.is_dir()])
for song in songs:
    for stem_name in STEMS:
            stem_path = song / stem_name
            if not stem_path.exists():
                continue
            y, sr = librosa.load(str(stem_path), sr=None)
            duration = librosa.get_duration(y=y, sr=sr)
            durations.append(duration)

# Compute mean duration
mean_duration = sum(durations) / len(durations)
print(f"\nTotal jazz files: {len(durations)}")
print(f"Mean duration (seconds): {mean_duration:.4f}")

In [ ]:
"""What are the unique sample rates present in the dataset? Enter
your answer as a comma separated list. Example: [40000, 50000, 60000]"""


unique_sample_rates = set()

for genre in GENRES:
    genre_path = STEMS_DIR / genre
    songs  = sorted([d for d in genre_path.iterdir() if d.is_dir()])
    for song in songs:
        for stem_name in STEMS:
            stem_path = song / stem_name
            if not stem_path.exists():
                continue
            try:
                info = sf.info(str(stem_path))
                unique_sample_rates.add(info.samplerate)
            except Exception as e:
                print(f"Error reading {audio_file.name}: {e}")

print("Unique sample rates:", sorted(unique_sample_rates))


In [ ]:
## How many corrupted or zero-byte audio files are present in the train dataset?

count_currupted=0

for genre in GENRES:
    genre_path = STEMS_DIR / genre
    songs      = sorted([d for d in genre_path.iterdir() if d.is_dir()])
    for song in songs:
        for stem_name in STEMS:
            stem_path = song / stem_name
            if not stem_path.exists():
                continue
            y, sr = librosa.load(str(stem_path), sr=None)
            duration = librosa.get_duration(y=y, sr=sr)
            if duration<=0:
                count_currupted+=1


print("total currupted fils is :", count_currupted)

In [ ]:
 ## What is the average peak amplitude (in dB) for vocal stems in train dataset? 

peak_amplitudes_db = []

for genre in GENRES:
    genre_path = STEMS_DIR / genre
    songs = sorted([d for d in genre_path.iterdir() if d.is_dir()])
    for song in songs:
        stem_path = song / "vocals.wav"
        if not stem_path.exists():
            continue
        try:
            y, sr = librosa.load(str(stem_path), sr=None)
            peak_linear = np.max(np.abs(y))
            
            # Convert to dB — avoid log(0) with a small epsilon
            peak_db = 20 * np.log10(peak_linear + 1e-9)
            peak_amplitudes_db.append(peak_db)
        except Exception as e:
            print(f"Error reading {stem_path}: {e}")

avg_peak_db = np.mean(peak_amplitudes_db)
print(f"Total vocal stems processed: {len(peak_amplitudes_db)}")
print(f"Average peak amplitude (dB): {avg_peak_db:.4f}")


In [ ]:
## What is the mean spectral centroid for 'blues' genre in the train dataset?  
blue_genre_path=STEMS_DIR/"blues"
print(" blues-path exist",blue_genre_path.exists())
songs = sorted([d for d in blue_genre_path.iterdir() if d.is_dir()])

all_spectral_centroids=[]
for song in songs:
    for stem_name in STEMS:
            stem_path = song / stem_name
            if not stem_path.exists():
                continue
            try: 
                y, sr = librosa.load(str(stem_path), sr=None)
                spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)

                # Mean across all frames for this stem
                mean_sc = np.mean(spectral_centroid)
                all_spectral_centroids.append(mean_sc)

            except Exception as e:
                print(f"Error reading {stem_path}: {e}")

# Final mean across ALL stems of ALL blues songs
overall_mean_sc = np.mean(all_spectral_centroids)
print(f"Total stems processed: {len(all_spectral_centroids)}")
print(f"Mean Spectral Centroid for 'blues': {overall_mean_sc:.4f} Hz")

In [ ]:
 # Which genre in the train dataset has the highest mean spectral centroid?  

genre_spectral_centroids = {} 
for genre in GENRES:
    genre_path = STEMS_DIR / genre
    songs      = sorted([d for d in genre_path.iterdir() if d.is_dir()])
    genre_sc_values = []
    for song in songs:
        for stem_name in STEMS:
            stem_path = song / stem_name
            if not stem_path.exists():
                continue
            try:
                y, sr = librosa.load(str(stem_path), sr=None)
                
                # Frame-wise spectral centroid → mean over frames
                sc = librosa.feature.spectral_centroid(y=y, sr=sr)
                genre_sc_values.append(np.mean(sc))
                
            except Exception as e:
                print(f"Error reading {stem_path}: {e}")
    
    genre_spectral_centroids[genre] = np.mean(genre_sc_values)
    print(f"{genre}: {genre_spectral_centroids[genre]:.4f} Hz")

In [ ]:
## How many stem audio files in the train dataset contain silence in the first 0.5 seconds?  
SILENCE_THRESHOLD_DB = -60.0  # dBFS
SILENCE_DURATION     = 0.5    # seconds

silent_files = []
total_files  = 0

for genre in GENRES:
    genre_path = STEMS_DIR / genre
    songs      = sorted([d for d in genre_path.iterdir() if d.is_dir()])
    genre_sc_values = []
    for song in songs:
        for stem_name in STEMS:
            stem_path = song / stem_name
            if not stem_path.exists():
                continue
            total_files += 1
            try:
                y, sr = librosa.load(str(stem_path), sr=None)
                # Extract first 0.5 seconds
                n_samples = int(SILENCE_DURATION * sr)  # 0.5 * 44100 = 22050 samples
                first_half_sec = y[:n_samples]
                # Peak amplitude of first 0.5s in dB
                peak_linear = np.max(np.abs(first_half_sec))
                peak_db     = 20 * np.log10(peak_linear + 1e-9)
                if peak_db < SILENCE_THRESHOLD_DB:
                    silent_files.append(str(stem_path))    
            except Exception as e:
                print(f"Error reading {stem_path}: {e}")

print(f"Total stems processed     : {total_files}")
print(f"Stems with silence in 0.5s: {len(silent_files)}")


### Milestone 2 part 2

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report

# --- 1. Setup and Preprocessing ---
ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

def extract_features(song_path):
    # Load 10s at 22050Hz
    y, sr = librosa.load(os.path.join(song_path, 'other.wav'), sr=22050, duration=10)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    return [float(tempo), spec_cent, zcr, rolloff]


# --- 2. Data Preparation & Stratified Split ---
data = []
for g in GENRES:
    gp = os.path.join(STEMS_PATH, g)
    songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
    for s in songs[:50]: # Sampling 50 for speed; use all for final
        data.append({'path': os.path.join(gp, s), 'genre': g})

df = pd.DataFrame(data)
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['genre'], random_state=42)

# --- 3. Model Training (Decision Tree) ---
X_train = np.array([extract_features(p) for p in train_df['path']])
y_train = train_df['genre']
X_val = np.array([extract_features(p) for p in val_df['path']])
y_val = val_df['genre']

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)


y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')
cm = confusion_matrix(y_val, y_pred, labels=GENRES)
cr = classification_report(y_val, y_pred, target_names=GENRES)


print(f"Validation Macro F1 Score: {macro_f1:.4f}\n")
print("Detailed Classification Report:")
print(cr)

# Visualize the confusion matrix
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=GENRES, yticklabels=GENRES,
            linewidths=0.5, linecolor='gray')
plt.title('Confusion Matrix — Decision Tree (max_depth=5)', fontsize=14)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()


#compute TP, TN, FP, FN for all genres.


print("\n--- Per-Genre TP / TN / FP / FN ---")
print(f"{'Genre':<12} {'TP':>6} {'TN':>6} {'FP':>6} {'FN':>6}")
print("-" * 40)

tp_tn_fp_fn = {}
for i, genre in enumerate(GENRES):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP   # predicted as genre, but wrong
    FN = cm[i, :].sum() - TP   # actual genre, but missed
    TN = cm.sum() - TP - FP - FN
    tp_tn_fp_fn[genre] = {'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN}
    print(f"{genre:<12} {TP:>6} {TN:>6} {FP:>6} {FN:>6}")

